In [1]:
import os

import uuid
import gradio as gr

from openai import OpenAI
from google import genai
from google.genai import types
from dotenv import load_dotenv

from io import BytesIO
from PIL import Image
from gtts import gTTS

import sqlite3
import json
import requests
import streamlit as st

In [12]:
# Groq LLM (reasoning brain)
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

MODEL = "llama-3.3-70b-versatile"

# Gemini (image + vision)
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

models = gemini_client.models.list()

for m in models:
    print(m.name)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [3]:
SYSTEM_PROMPT = """
You are a Greeting Card Creative Assistant.

Your job:
1. Help user design happy greeting cards.
2. Always suggest structure first unless final request is explicit.
3. Keep tone emotional, warm, celebratory.
4. Never produce negative or harmful content.
"""

In [4]:
def get_text_response(user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

In [13]:
def generate_card_image(prompt):
    response = gemini_client.models.generate_content(
        model="models/gemini-2.5-flash-image",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"]
        )
    )

    for candidate in response.candidates:
        for part in candidate.content.parts:
            if part.inline_data:
                return Image.open(BytesIO(part.inline_data.data))

    raise Exception("Image generation failed")

In [6]:
def analyze_reference_image(image):
    response = gemini_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=["Describe this greeting card style in detail", image]
    )
    return response.text

In [7]:
def generate_audio(text):
    filename = f"card_audio_{uuid.uuid4().hex}.mp3"

    tts = gTTS(text=text, lang="en")
    tts.save(filename)

    return filename

In [8]:
def card_agent(user_text, image=None):
    
    result = {}

    # Step 1: Text reasoning
    text_response = get_text_response(user_text)
    result["text"] = text_response

    # Step 2: Image generation prompt refinement
    image_prompt = f"""
    A beautiful greeting card design for:
    {user_text}

    Style: happy, colorful, emotional, minimal text space
    """

    result["image"] = generate_card_image(image_prompt)

    # Step 3: Audio narration
    result["audio"] = generate_audio(text_response)

    return result

In [ ]:
def run_agent(message, image):
    output = card_agent(message, image)

    return (
        output["text"],
        output["image"],
        output["audio"]
    )


demo = gr.Interface(
    fn=run_agent,
    inputs=[
        gr.Textbox(label="Describe your greeting card"),
        gr.Image(type="pil", label="Optional Reference Image")
    ],
    outputs=[
        gr.Textbox(label="Card Message"),
        gr.Image(label="Generated Card"),
        gr.Audio(label="Voice Message")
    ],
    title="Multimodal Greeting Card Agent"
)

demo.launch()